# COMPAS Racial Bias: Reproducing the Finding and Mitigating It

## Introduction

COMPAS (Correctional Offender Management Profiling for Alternative Sanctions) is a proprietary risk-assessment algorithm developed by Northpointe (now Equivant) and deployed across the United States criminal justice system. The tool ingests demographic and criminal-history data about a defendant and outputs a recidivism risk score—typically a number from 1 to 10—that categorizes individuals as low, medium, or high risk. These scores were not advisory footnotes. Judges in Florida, Wisconsin, New York, and elsewhere incorporated them directly into sentencing and bail decisions, meaning a number generated by a black-box model could determine whether a human being went home that night or was locked in a cell.

In 2016, ProPublica journalists analyzed more than 7,000 defendants processed through Broward County, Florida's courts and published *Machine Bias*, one of the most consequential pieces of data journalism of the decade. Their central finding: COMPAS was calibrated to be roughly equally accurate across racial groups in the aggregate, yet it failed in systematically asymmetric ways. Black defendants who did *not* reoffend were nearly twice as likely as white defendants to be falsely flagged as high risk (false positive rate: ~45% vs. ~24%). White defendants who *did* reoffend were more likely than Black defendants to be incorrectly labeled low risk (false negative rate: ~48% vs. ~28%). The algorithm's errors were not random noise—they fell along racial lines in ways that systematically disadvantaged Black defendants at the moment of sentencing.

This notebook does three things. First, it reproduces ProPublica's core finding from the public Broward County dataset, validating that the racial disparity in false positive and false negative rates is real and statistically robust—not a reporting artifact. Second, it quantifies that disparity rigorously using confidence intervals and hypothesis tests, so the magnitude of harm is expressed with appropriate uncertainty rather than as a single alarming headline number. Third, it applies a fairness-aware post-processing pipeline (`fairpipe`) to the same data, re-calibrates the score thresholds to equalize error rates across groups, and measures how much of the disparity is recoverable—and at what cost to overall predictive accuracy.

In [ ]:
# Uncomment if running in Colab or Binder
# !pip install fairpipe

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fairpipe import FairnessAnalyzer, load_data
from fairpipe.pipeline import load_config, build_pipeline, apply_pipeline, run_detectors
from fairpipe.integration import execute_workflow
 
THRESHOLD = 0.05

In [ ]:
RAW_URL = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
raw = pd.read_csv(RAW_URL)
 
# Apply ProPublica's documented cleaning criteria
df = raw[
    (raw["days_b_screening_arrest"] <= 30) &
    (raw["days_b_screening_arrest"] >= -30) &
    (raw["is_recid"] != -1) &
    (raw["c_charge_degree"] != "O") &
    (raw["score_text"] != "N/A")
].reset_index(drop=True)
 
print(f"Dataset: {len(df):,} defendants")
print(f"Recidivism rate: {df['two_year_recid'].mean():.1%}")
print(f"\nRacial breakdown:\n{df['race'].value_counts()}")

In [ ]:
# Convert COMPAS score to binary prediction (score >= 5 = predicted high risk)
# This mirrors how the score is used in practice and how ProPublica operationalised it
df["y_pred"] = (df["decile_score"] >= 5).astype(int)
df["y_true"] = df["two_year_recid"]

# Focus on the two largest racial groups for statistical power
df_bw = df[df["race"].isin(["African-American", "Caucasian"])].copy()
 
print(f"Analysis subset: {len(df_bw):,} defendants")
print(f"African-American: {(df_bw['race']=='African-American').sum():,}")
print(f"Caucasian: {(df_bw['race']=='Caucasian').sum():,}")

## Measure the Bias


In [ ]:
THRESHOLD = 0.05

analyzer = FairnessAnalyzer.from_dataframe(
    df_bw,
    y_pred_col="y_pred",
    y_true_col="y_true",
    sensitive_col="race",
    min_group_size=30, # ensure we have enough data for reliable estimates
)

# Demographic parity difference 
dpd = analyzer.demographic_parity_difference(with_ci=True)
passed_dpd = dpd.value <= THRESHOLD


print(f"Demographic Parity Difference: {dpd.value:.4f} | Threshold: {THRESHOLD}")
print(f"95% CI: [{dpd.ci[0]:.4f}, {dpd.ci[1]:.4f}]")
print(f"Group prediction rates: {dpd.n_per_group}")
print(f"\n{'✅ PASSED' if passed_dpd else '❌ FAILED — pipeline would be blocked'}")

print()

# Equalized odds (captures false positive rate disparity)
eod = analyzer.equalized_odds_difference(with_ci=True)
passed_eod = eod.value <= THRESHOLD
print(f"Equalized Odds Difference: {eod.value:.4f} | Threshold: {THRESHOLD}")
print(f"95% CI: [{eod.ci[0]:.4f}, {eod.ci[1]:.4f}]")
print(f"\n{'✅ PASSED' if passed_eod else '❌ FAILED — pipeline would be blocked'}")


The COMPAS algorithm scores Black defendants as high-risk at a rate **24.5
percentage points higher** than white defendants (DPD = 0.2451,
95% CI: [0.2177, 0.2700]). 

This gap is not explained by actual recidivism
rates. The Equalized Odds Difference of **0.2116** (95% CI: [0.1882, 0.2560])
tells the more troubling story: among defendants who will *not* reoffend, a
Black defendant is **21 percentage points more likely** to be incorrectly
labelled high-risk than a white defendant in the same situation. 

In practice
this means a person who poses no risk loses their liberty at a significantly
higher rate depending on their race. Both confidence intervals sit entirely
above zero — this is not a statistical artefact. With 3,175 Black defendants
and 2,103 white defendants in the analysis, the sample is large enough that
these findings are precise to within roughly ±2.5 percentage points.

## Visualize the Bias

In [ ]:
# False positive and false negative rates by race
groups = df_bw.groupby("race")
rates = {}

for name, group in groups:
    fp = ((group["y_pred"]==1) & (group["y_true"]==0)).sum() / (group["y_true"]== 0).sum()
    fn = ((group["y_pred"]==0) & (group["y_true"]==1)).sum() / (group["y_true"]==1).sum()
    rates[name] = {"False Positive Rate":fp, "False Negative Rate": fn}

rates_df = pd.DataFrame(rates).T
ax = rates_df.plot(kind="bar", figsize=(8,4), color=["#d62728", "#1f77b4"], rot=0)
ax.set_title("COMPAS Error Rates by race\n(False Positive = flagged high risk, did not recidivate; False Negative = flagged low risk, did recidivate)")
ax.set_ylabel("Rate")
ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.savefig("compas_error_rates.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Saving analysis subset for CLI demo
df_bw[["y_pred", "y_true", "race"]].to_csv("compas_bw.csv", index=False)


In [ ]:
%%bash
fairpipe validate \
  --csv compas_bw.csv \
  --y-true y_true \
  --y-pred y_pred \
  --sensitive race \
  --with-ci \
  --with-effects \
  --threshold 0.05 \
  --metric equalized_odds_difference \
  --out compas_validation_report.md 2>/dev/null

echo "Exit code: $?"
cat compas_validation_report.md

**Exit code 1** - the CLI detected an EOD of 0.2116 against a threshold of 0.05 and failed the pipeline. In a CI/CD workflow this commit would be blocked from merging. The effect size of **1.92** on equalized odds difference is classified as large by Cohen's conventions, meaning the disparity is not only statistically significant but also practically meaningful. Not just a marginal finding, but a real-world problem that demands attention.

## Step 2: Detecting the Bias
 
Measuring bias is necessary but not sufficient. The question practitioners
actually face is: *what do we do about it?*
 
fairpipe's pipeline module provides a set of bias mitigation transformers that
operate directly on your data before model training. Here we apply **Instance
Reweighting** — one of the most well-established pre-processing mitigation
techniques. It works by assigning sample weights that reduce the influence of
over-represented group/label combinations, nudging the model toward more
equitable predictions without discarding any data.
 
Before applying the transformer, we run fairpipe's **bias detectors** — a
suite of statistical tests that scan the dataset for representation imbalance,
proxy variables, and distributional disparities across sensitive groups.
 
The `features` key in the config explicitly specifies which columns are used
for model training, excluding identifiers, dates, and high-strength proxy
variables.

In [ ]:
from fairpipe.pipeline import load_config, build_pipeline, apply_pipeline, run_detectors

# write a minimal config 
config_yaml = """
sensitive: ["race"]
features: ["age", "priors_count", "juv_fel_count", "juv_misd_count", "juv_other_count"]
pipeline:
  - name: reweigh
    transformer: "InstanceReweighting"
training:
  method: "reductions"
  target_column: "y_true"
  params:
    constraint: "demographic_parity"
    eps: 0.05
fairness_metric: "equalized_odds_difference"
validation_threshold: 0.05
"""

with open("compas_config.yml", "w") as f:
    f.write(config_yaml)

config = load_config("compas_config.yml")

# Detect bias 
detector_report = run_detectors(df=df_bw, cfg=config)
summary = detector_report.body["summary"]
print("===== Bias Detection Summary ===== ")
print(f"Sensitive attribute: race")
print(f"Disparity flags: {summary['disparity_flags']} (features with significant racial bias)")
print(f"Proxy flags: {summary['proxy_flags']} (features that are strong proxies for race)")
print(f"Representation flags: {summary['representation_flags']} (group size imbalances)")

# Highlight the most import flagged proxies by strength 
proxies = detector_report.body["proxies"]
strong_proxies = sorted(
    [p for p in proxies if p["flagged"]],
    key=lambda x: x["strength"],
    reverse=True
)[:5]

print("\n --- Top 5 Strongest Proxy Features ---")
for p in strong_proxies:
    print(f" {p['feature']:<30} Cramer's V = {p['strength']:.3f}")




The detector found **28 features** with statistically significant racial
disparities** and **23 potential proxy variables** — features that are not
race but are strongly correlated with it and could allow a model to
discriminate indirectly.
 
The top proxies (Cramér's V ≈ 1.0) are identifiers like case numbers, jail
dates, and names — essentially unique keys that encode race perfectly. A model
trained on these features would learn racial patterns even if the `race` column
were removed. This is the proxy discrimination problem in its clearest form.
 
Instance Reweighting addresses this by rebalancing the influence of each sample
during training. In a production pipeline you would also invoke `ProxyDropper`
to remove the highest-strength proxies before training.


## Step 3: Mitigation and Validation via execute_workflow
 
`execute_workflow` runs the complete three-step pipeline automatically:
 
1. **Baseline measurement** — trains a logistic regression on the specified
   features and measures fairness on the held-out test set
2. **Reweighting** — applies Instance Reweighting and trains a second model
   using the corrected sample weights
3. **Validation** — compares before/after metrics against the threshold
   defined in the config
 
This is the intended production workflow. A single config file, a single
function call, a clear pass/fail verdict.

In [ ]:
result = execute_workflow(
    config=config,
    df=df_bw,
    output_dir=None,
    min_group_size=30,
    train_size=0.8
)

vr = result.validation_result

improvement_pct = (abs(vr.improvement) / vr.baseline_metric_value) * 100

print("=== Before vs After Mitigation (execute_workflow) ===")
print(f"{'Metric':<6}  {'Before':>8}  {'After':>8}  {'Change':>8}  {'Threshold':>10}  {'Status'}")
print(f"{'-'*62}")
print(f"{'EOD':<6}  {vr.baseline_metric_value:>8.4f}  {vr.final_metric_value:>8.4f}  "
      f"{vr.final_metric_value - vr.baseline_metric_value:>+8.4f}  {THRESHOLD:>10.2f}  "
      f"{'✅ PASSED' if vr.passed else '❌ FAILED'}")
print()
print(f"Improvement:                       {improvement_pct:.1f}%")
print(f"Remaining gap to threshold:        {max(0, vr.final_metric_value - THRESHOLD):.4f}")
print()
print(f"Pipeline status BEFORE mitigation: ❌ FAILED")
print(f"Pipeline status AFTER  mitigation: {'✅ PASSED' if vr.passed else '❌ FAILED'}")
print(f"\nMessage: {vr.message}")

### What this tells us

`execute_workflow` ran the complete pipeline - baseline measurement,
instance reweighting with correctly applied sample weights, and fair model
training — in a single call. The results:

- EOD dropped from **0.2083 to 0.0960**, a reduction of **0.1123 (53.9%)**
- The remaining gap to the 0.05 threshold is just **0.0460**
- The pipeline correctly reports **❌ FAILED** - substantial progress
  was made but the bias was not fully resolved

A 53.9% reduction in Equalized Odds Difference is a strong result for a
single pre-processing intervention on five numeric features. The model is
now less than 5 percentage points from clearing the fairness threshold.

The remaining gap reflects the structural nature of the problem. Features
like `priors_count` and `age` carry racial signal that reweighting alone
cannot fully eliminate. Closing the final 0.0460 gap would require one
or more of:

- **Proxy removal** — using `ProxyDropper` to drop the highest-strength
  proxy features before training
- **Stronger constraints** — using `ReductionsWrapper` with an explicit
  equalized odds constraint
- **Accepting a performance-fairness tradeoff** — a fairer model on this
  data will have lower overall accuracy

This is not a failure of fairpipe. It is fairpipe working exactly as
intended i.e. surfacing a hard problem clearly so practitioners can make
an informed decision, rather than shipping a biased model unknowingly.

## From Notebook to CI/CD Pipeline
 
Running this analysis manually in a notebook is valuable for understanding.
But in a production ML team, fairness checks need to happen automatically —
on every pull request, before every deployment, without relying on anyone
remembering to run a notebook.
 
This is what the `fairpipe` GitHub Action does. Add three lines of YAML to
your repository and every PR is automatically checked against your fairness
threshold:
 
```yaml
- uses: SvrusIO/fairpipe-action@v1
  with:
    csv: data/predictions.csv
    y-true: y_true
    y-pred: y_pred
    sensitive: race
    threshold: "0.05"
    metric: "equalized_odds_difference"
    fail-on-violation: "true"
```
 
If the Equalized Odds Difference exceeds 0.05 the PR is blocked. The same
metric that flagged a 0.2116 disparity in this notebook would have blocked
every COMPAS model deployment automatically — before it reached a judge's
courtroom.
 
The action writes a full fairness report to the GitHub Actions job summary,
including metric values, confidence intervals, and group breakdowns, so the
evidence is attached to the commit permanently.
 
→ **[SvrusIO/fairpipe-action](https://github.com/SvrusIO/fairpipe-action)**


## Conclusions

This notebook reproduced and extended ProPublica's 2016 finding using
`fairpipe` as the measurement and mitigation framework. The key results:

- The COMPAS algorithm scores African-American defendants as high-risk at a rate
  **24.5 percentage points higher** than white defendants (DPD = 0.2451,
  95% CI: [0.2177, 0.2700])
- Among defendants who will not reoffend, African-American defendants are **21.2
  percentage points more likely** to be incorrectly labelled high-risk
  (EOD = 0.2116, 95% CI: [0.1882, 0.2560])
- Both confidence intervals sit entirely above zero — this is not noise
- Both metrics failed against a 0.05 threshold — in a CI/CD pipeline
  this model would be blocked from deployment
- The bias detector identified **28 features** with statistically significant
  racial disparities** and **23 proxy variables** — removing the `race`
  column alone would not fix this model
- `execute_workflow` ran the complete mitigation pipeline end-to-end,
  reducing EOD from **0.2083 to 0.0960** — a **53.9% improvement**,
  bringing the model to within 0.0460 of the 0.05 fairness threshold
- Closing the gap fully requires stronger interventions: proxy removal,
  constraint-based training, or accepting a performance-fairness tradeoff

The deeper point is not about COMPAS specifically. It is that any model
making decisions about people — in hiring, lending, healthcare, or criminal
justice — can carry disparities this large without any single person
intending it. The question is whether your team will find out before or
after deployment.

`fairpipe` is built to make sure you find out before.

---

### Try It Yourself

[![Launch in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/SvrusIO/fAIr/main?urlpath=%2Fdoc%2Ftree%2Fcase_studies%2Fcompas_racial_bias.ipynb)

```bash
pip install fairpipe
```

**→ [GitHub](https://github.com/SvrusIO/fAIr) · [PyPI](https://pypi.org/project/fairpipe/) · [GitHub Action](https://github.com/SvrusIO/fairpipe-action)**

*Built by [Svrus](https://github.com/SvrusIO) — Responsible Intelligence for Action*